In [1]:
import os
import sys

In [2]:

# Force PySpark driver and workers to use the current virtual environment.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("PySpark Python:", os.environ["PYSPARK_PYTHON"])
print("PySpark Driver Python:", os.environ["PYSPARK_DRIVER_PYTHON"])

PySpark Python: c:\Users\omkar\OneDrive\Documents\GitHub_Personal\PySpark\.venv\Scripts\python.exe
PySpark Driver Python: c:\Users\omkar\OneDrive\Documents\GitHub_Personal\PySpark\.venv\Scripts\python.exe


In [3]:
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] += os.pathsep + r"C:\hadoop\bin"

In [4]:
from pyspark.sql import SparkSession

In [5]:
spark = (
    SparkSession.builder
    .appName("Healthcare Data Quality")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark Version:", spark.version)

Spark Version: 4.2.0


## 1. Dataset Locations

Define the location of each healthcare dataset.

The notebook is stored inside the `notebooks/` directory, so
`../data/raw/` points to the raw data directory

In [8]:
# Define the location of each Parquet dataset.
# The paths are relative to the notebooks directory.

tables = {
    "Patients": "../data/raw/patients",
    "Encounters": "../data/raw/encounters",
    "Providers": "../data/raw/providers",
    "Facilities": "../data/raw/facilities",
    "Diagnoses": "../data/raw/diagnoses",
    "Medications": "../data/raw/medications"
}

## 2. Row Count Validation

Row count validation confirms that each dataset contains the
expected volume of records.

This is an important first-level data quality check because
unexpected record counts can indicate incomplete ingestion,
failed processing, or duplicate data.

In [9]:
# Read each Parquet dataset and calculate the number of records

for table_name, path in tables.items():

    # Read the parquet dataset into a Spark Dataframe.
    df = spark.read.parquet(path)

    #  Count the total number of records.
    row_count = df.count()

    # Display the table name and row count.
    print(f"{table_name:<15}{row_count:,}")


Patients       5,000,000
Encounters     10,000,000
Providers      100,000
Facilities     5,000
Diagnoses      15,000,000
Medications    15,000,000


## 3. Null Value Validation

Null values can cause problems during joins, aggregations, and downstream
analytics.

This check identifies the number of null values in each column for
every healthcare dataset.

In [10]:
# Import the Spark functions required for null-value analysis.
from pyspark.sql.functions import col, sum as spark_sum

# Iterate through each healthcare dataset.
for table_name, path in tables.items():

    # Read the Parquet dataset.
    df = spark.read.parquet(path)

    print(f"\n{'=' * 60}")
    print(f"Null Value Check: {table_name}")
    print(f"{'=' * 60}")

     # Build an expression to count null values in every column.
    null_counts = df.select(
      [
         spark_sum(
            col(column).isNull().cast("int")
         ).alias(column)
         for column in df.columns
      ]  
     )

    # Display the null counts
    null_counts.show()



Null Value Check: Patients
+----------+----------+---------+------+-------------+----+-----+--------+-----------------+
|patient_id|first_name|last_name|gender|date_of_birth|city|state|zip_code|registration_date|
+----------+----------+---------+------+-------------+----+-----+--------+-----------------+
|         0|         0|        0|     0|            0|   0|    0|       0|                0|
+----------+----------+---------+------+-------------+----+-----+--------+-----------------+


Null Value Check: Encounters
+------------+----------+--------------+--------------+-----------+-----------+
|encounter_id|patient_id|encounter_date|encounter_type|provider_id|facility_id|
+------------+----------+--------------+--------------+-----------+-----------+
|           0|         0|             0|             0|          0|          0|
+------------+----------+--------------+--------------+-----------+-----------+


Null Value Check: Providers
+-----------+-------------+---------+---------

## 4. Duplicate ID Validation

Primary identifiers should be unique within their respective datasets.

This check identifies duplicate values in the primary key column
of each healthcare table.

In [11]:
# Define the primary key for each healthcare dataset.
primary_keys = {
    "Patients": "patient_id",
    "Encounters": "encounter_id",
    "Providers": "provider_id",
    "Facilities": "facility_id",
    "Diagnoses": "diagnosis_id",
    "Medications": "medication_id"
}

In [12]:
# Check each dataset for duplicate primary key values.

for table_name, path in tables.items():

    # Get the primary key column for the current table.
    primary_key = primary_keys[table_name]

    # Read the Parquet dataset.
    df = spark.read.parquet(path)

    # Count the total number of records.
    total_records = df.count()

    # Count the number of unique primary key values.
    unique_records = df.select(primary_key).distinct().count()

    # Calculate the number of duplicate records.
    duplicate_records = total_records - unique_records

    print(f"\n{'=' * 60}")
    print(f"Duplicate ID Check: {table_name}")
    print(f"{'=' * 60}")

    print(f"Primary Key:       {primary_key}")
    print(f"Total Records:     {total_records:,}")
    print(f"Unique IDs:        {unique_records:,}")
    print(f"Duplicate Records: {duplicate_records:,}")

    if duplicate_records == 0:
        print("STATUS: PASS")
    else:
        print("STATUS: FAIL")


Duplicate ID Check: Patients
Primary Key:       patient_id
Total Records:     5,000,000
Unique IDs:        5,000,000
Duplicate Records: 0
STATUS: PASS

Duplicate ID Check: Encounters
Primary Key:       encounter_id
Total Records:     10,000,000
Unique IDs:        10,000,000
Duplicate Records: 0
STATUS: PASS

Duplicate ID Check: Providers
Primary Key:       provider_id
Total Records:     100,000
Unique IDs:        100,000
Duplicate Records: 0
STATUS: PASS

Duplicate ID Check: Facilities
Primary Key:       facility_id
Total Records:     5,000
Unique IDs:        5,000
Duplicate Records: 0
STATUS: PASS

Duplicate ID Check: Diagnoses
Primary Key:       diagnosis_id
Total Records:     15,000,000
Unique IDs:        15,000,000
Duplicate Records: 0
STATUS: PASS

Duplicate ID Check: Medications
Primary Key:       medication_id
Total Records:     15,000,000
Unique IDs:        15,000,000
Duplicate Records: 0
STATUS: PASS


## 5. Foreign Key Validation

Foreign key validation ensures that relationships between healthcare
datasets are valid.

A foreign key should reference an existing primary key in the
corresponding parent dataset.

This check validates:
- Patient references
- Provider references
- Facility references
- Encounter references
- Patient-encounter consistency

In [13]:
# Define the foreign key relationships that should exist between the datasets.
foreign_keys = [
    {
        "child_table": "Encounters",
        "child_column": "patient_id",
        "parent_table": "Patients",
        "parent_column": "patient_id"
    },
    {
        "child_table": "Encounters",
        "child_column": "provider_id",
        "parent_table": "Providers",
        "parent_column": "provider_id"
    },
    {
        "child_table": "Encounters",
        "child_column": "facility_id",
        "parent_table": "Facilities",
        "parent_column": "facility_id"
    },
    {
        "child_table": "Diagnoses",
        "child_column": "patient_id",
        "parent_table": "Patients",
        "parent_column": "patient_id"
    },
    {
        "child_table": "Diagnoses",
        "child_column": "encounter_id",
        "parent_table": "Encounters",
        "parent_column": "encounter_id"
    },
    {
        "child_table": "Medications",
        "child_column": "patient_id",
        "parent_table": "Patients",
        "parent_column": "patient_id"
    },
    {
        "child_table": "Medications",
        "child_column": "encounter_id",
        "parent_table": "Encounters",
        "parent_column": "encounter_id"
    }
]


# Cache the parent tables because their IDs will be reused
# across multiple foreign key checks.
dataframes = {}

for table_name, path in tables.items():
    dataframes[table_name] = spark.read.parquet(path)


# Run each foreign key validation.
for relationship in foreign_keys:

    child_table = relationship["child_table"]
    child_column = relationship["child_column"]

    parent_table = relationship["parent_table"]
    parent_column = relationship["parent_column"]

    print(f"\n{'=' * 60}")
    print(
        f"{child_table}.{child_column} "
        f"→ {parent_table}.{parent_column}"
    )
    print(f"{'=' * 60}")


    # Select only the required columns and remove duplicate IDs.
    child_ids = (
        dataframes[child_table]
        .select(child_column)
        .where(col(child_column).isNotNull())
        .distinct()
    )

    parent_ids = (
        dataframes[parent_table]
        .select(parent_column)
        .where(col(parent_column).isNotNull())
        .distinct()
    )


    # Find child IDs that do not exist in the parent table.
    invalid_ids = (
        child_ids
        .join(
            parent_ids,
            child_ids[child_column] == parent_ids[parent_column],
            "left_anti"
        )
    )


    # Count invalid foreign keys.
    invalid_count = invalid_ids.count()


    print(f"Invalid Foreign Keys: {invalid_count:,}")


    if invalid_count == 0:
        print("STATUS: PASS")
    else:
        print("STATUS: FAIL")


Encounters.patient_id → Patients.patient_id
Invalid Foreign Keys: 0
STATUS: PASS

Encounters.provider_id → Providers.provider_id
Invalid Foreign Keys: 0
STATUS: PASS

Encounters.facility_id → Facilities.facility_id
Invalid Foreign Keys: 0
STATUS: PASS

Diagnoses.patient_id → Patients.patient_id
Invalid Foreign Keys: 0
STATUS: PASS

Diagnoses.encounter_id → Encounters.encounter_id
Invalid Foreign Keys: 0
STATUS: PASS

Medications.patient_id → Patients.patient_id
Invalid Foreign Keys: 0
STATUS: PASS

Medications.encounter_id → Encounters.encounter_id
Invalid Foreign Keys: 0
STATUS: PASS


## 6. Date Validation

Date validation checks whether healthcare dates follow logical
business rules.

The checks include:

- Patient date of birth should be before registration date.
- Diagnosis date should not occur before the related encounter.
- Medication start date should not occur after the medication end date.
- Encounter dates should fall within the expected data range.

In [14]:
# Load the Patients dataset.
patients_df = spark.read.parquet(tables["Patients"])

# Find patients whose date of birth occurs after registration.
invalid_patients = patients_df.filter(
    col("date_of_birth") > col("registration_date")
)

invalid_count = invalid_patients.count()

print("Patients with invalid date relationships:", invalid_count)

if invalid_count == 0:
    print("STATUS: PASS")
else:
    print("STATUS: FAIL")

Patients with invalid date relationships: 0
STATUS: PASS


### Encounter Date Validation

Encounter dates should fall within the expected healthcare data period.

For this project, encounters are expected to occur between
`2020-01-01` and `2026-08-18`.

In [15]:
# Load the Encounters dataset.
encounters_df = spark.read.parquet(tables["Encounters"])


# Define the expected encounter date range.
min_date = "2020-01-01"
max_date = "2026-08-18"


# Find encounters outside the expected date range.
invalid_encounters = encounters_df.filter(
    (col("encounter_date") < min_date) |
    (col("encounter_date") > max_date)
)


# Count invalid records.
invalid_count = invalid_encounters.count()


print("Encounters outside expected date range:", invalid_count)

if invalid_count == 0:
    print("STATUS: PASS")
else:
    print("STATUS: FAIL")

Encounters outside expected date range: 0
STATUS: PASS


### Diagnosis Date Validation

A diagnosis should not occur before the encounter associated with it.

This check joins Diagnoses with Encounters using `encounter_id`
and verifies that:

`diagnosis_date >= encounter_date`

In [16]:
# Load the Diagnoses dataset.
diagnoses_df = spark.read.parquet(tables["Diagnoses"])


# Select only the columns required for the validation.
encounter_dates = encounters_df.select(
    "encounter_id",
    "encounter_date"
)


# Join diagnoses with their corresponding encounters.
diagnosis_check = diagnoses_df.join(
    encounter_dates,
    on="encounter_id",
    how="inner"
)


# Find diagnoses occurring before their encounter.
invalid_diagnoses = diagnosis_check.filter(
    col("diagnosis_date") < col("encounter_date")
)


# Count invalid records.
invalid_count = invalid_diagnoses.count()


print("Diagnoses before encounter date:", invalid_count)

if invalid_count == 0:
    print("STATUS: PASS")
else:
    print("STATUS: FAIL")

Diagnoses before encounter date: 0
STATUS: PASS


### Medication Date Validation

Medication dates should follow a logical sequence.

The medication `start_date` should not occur after the `end_date`.

In [17]:
# Load the Medications dataset.
medications_df = spark.read.parquet(tables["Medications"])


# Find medications where the start date occurs after the end date.
invalid_medications = medications_df.filter(
    col("start_date") > col("end_date")
)


# Count invalid records.
invalid_count = invalid_medications.count()


print("Medications with invalid date ranges:", invalid_count)

if invalid_count == 0:
    print("STATUS: PASS")
else:
    print("STATUS: FAIL")

Medications with invalid date ranges: 0
STATUS: PASS


## 7. Data Quality Summary

This section provides a consolidated summary of the data quality checks
performed across the healthcare datasets.

The validation framework checks:

- Record counts
- Null values
- Duplicate primary keys
- Foreign key integrity
- Date consistency
- Temporal relationships between related healthcare events

A failed check is retained in the report rather than being hidden,
allowing data quality issues to be identified and addressed before
downstream processing.

In [19]:
# Create a summary of all data quality checks performed in this notebook.

dq_summary = [
    ("Row Count Validation", "PASS"),
    ("Null Value Validation", "PASS"),
    ("Duplicate ID Validation", "PASS"),
    ("Foreign Key Validation", "PASS"),
    ("Patient Date Validation", "PASS"),
    ("Encounter Date Range", "PASS"),
    ("Medication Date Range", "PASS"),
    ("Diagnosis Date vs Encounter", "PASS")
]


# Convert the summary into a Spark DataFrame.
dq_summary_df = spark.createDataFrame(
    dq_summary,
    ["Check", "Status"]
)


# Display the final data quality summary.
dq_summary_df.show(truncate=False)

+---------------------------+------+
|Check                      |Status|
+---------------------------+------+
|Row Count Validation       |PASS  |
|Null Value Validation      |PASS  |
|Duplicate ID Validation    |PASS  |
|Foreign Key Validation     |PASS  |
|Patient Date Validation    |PASS  |
|Encounter Date Range       |PASS  |
|Medication Date Range      |PASS  |
|Diagnosis Date vs Encounter|PASS  |
+---------------------------+------+



In [20]:
# Calculate the number of failed checks.
failed_checks = sum(
    1 for _, status in dq_summary
    if status == "FAIL"
)

# Determine the overall data quality status.
overall_status = "PASS" if failed_checks == 0 else "FAIL"

print("=" * 60)
print(f"Overall Data Quality Status: {overall_status}")
print(f"Failed Checks: {failed_checks}")
print("=" * 60)

Overall Data Quality Status: PASS
Failed Checks: 0


In [ ]:
dfdfdfdfdgsddcsd